# Phase 6: Summarisation: GQA decoder

This is the **Summarisation** part of the full `gqa` notebook (`notebooks/CS781_gqa_all_phases.ipynb`), extracted so it can be read on its own. It reuses the tokenizer, model classes and helpers defined in the earlier cells, so **to run it, execute the full notebook from the top** (or run the Phase 1 set-up cells first, then this part).

## Abstractive Summarization

In [ ]:
import json, os, numpy as np
from huggingface_hub import snapshot_download
P6_DIR = "/content/drive/MyDrive/CS781_Capstone_Project_Phase_6"; os.makedirs(P6_DIR, exist_ok=True)
local = snapshot_download(repo_id=HF_DATASET_REPO, repo_type="dataset",
                          allow_patterns=["Phase_6_Summ/*", "*/Phase_6_Summ/*"], token=HF_TOKEN)
def find(name):
    for r, _, fs in os.walk(local):
        if name in fs and "Phase_6" in r: return os.path.join(r, name)
def rows(d):
    if isinstance(d, list): return d
    if all(isinstance(v, dict) for v in d.values()): return [dict(id=k, **v) for k, v in d.items()]
    return [dict(zip(d.keys(), v)) for v in zip(*d.values())]
tr6, te6 = rows(json.load(open(find("train.json")))), rows(json.load(open(find("test.json"))))
print("train:", len(tr6), "| test:", len(te6), "| train keys:", list(tr6[0].keys()), "| test keys:", list(te6[0].keys()))
dk = "document" if "document" in tr6[0] else "text"
print("\nDOC start :", tr6[0][dk][:300].replace("\n", " "))
print("SUMMARY   :", tr6[0]["summary"][:300].replace("\n", " "))
print("duplicate ids:", len(tr6) - len({r["id"] for r in tr6}), "| empty summaries:", sum(not r["summary"].strip() for r in tr6))

def toklen(texts):
    return np.array([len(e.ids) for e in tokenizer.encode_batch([t[:300000] for t in texts])])
rng = np.random.RandomState(0); samp = [tr6[i] for i in rng.permutation(len(tr6))[:500]]
dl, sl = toklen([r[dk] for r in samp]), toklen([r["summary"] for r in samp])
tl = toklen([r[dk if dk in r else "document"] for r in [te6[i] for i in rng.permutation(len(te6))[:500]]])
q = lambda a: {p: int(np.percentile(a, p)) for p in (10, 50, 90, 99)} | {"max": int(a.max())}
print("\ndocument tokens (500 train docs):", q(dl))
print("summary  tokens (500 train docs):", q(sl))
print("test document tokens (500 docs) :", q(tl))
print("compression ratio (median doc/summary): %.1fx" % np.median(dl / np.maximum(sl, 1)))
for L in (1024, 2048, 3072, 4096): print(f"docs <= {L} tokens: {100 * (dl <= L).mean():.1f}%")
for L in (128, 256, 320, 512): print(f"summaries <= {L} tokens: {100 * (sl <= L).mean():.1f}%")

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

train: 7030 | test: 100 | train keys: ['id', 'document', 'summary'] | test keys: ['id', 'document']

DOC start : IVIL Appeal No.768 (NT) of 1977 etc. From the judgment Order dated 9.10.1975 of the Madhya Pradesh High Court in M.C.C. No.144 of 1966. V.A. Bobde, B.R.Aggarwala and U.A.Rana for the Appellant. S.V.Deshpande and S.K.Agnihotri for the Respondents. The Judgment of the Court was delivered by RANGANATH 
SUMMARY   : The appellant, a manufacturer of cement, entered into an agreement with the Cement Manufacturing Company of India Limited, for sale of cement. Under the agreement, the appellant was to sell its cement only through the marketing company, and certain sums would be paid for the cement supplied by the m
duplicate ids: 0 | empty summaries: 2

document tokens (500 train docs): {10: 1673, 50: 3699, 90: 9867, 99: 34721, 'max': 51763}
summary  tokens (500 train docs): {10: 352, 50: 773, 90: 1833, 99: 5569, 'max': 9803}
test document tokens (500 docs) : {10: 1766, 50: 3754, 90:

In [ ]:
import os, json, pickle, numpy as np
from huggingface_hub import snapshot_download
P6_DIR = "/content/drive/MyDrive/CS781_Capstone_Project_Phase_6"; os.makedirs(P6_DIR, exist_ok=True)
CACHE6 = f"{P6_DIR}/sum_tok_cache.pkl"
EOS6 = tokenizer.token_to_id("<|endoftext|>")

def doc_view(rec, n_head, n_tail=0):
    """Token view of a document: first n_head tokens, plus last n_tail tokens if the doc is longer."""
    n, h, t = rec["n"], rec["head"], rec["tail"]
    full = h if n <= 4096 else (np.concatenate([h, t[-(n - 4096):]]) if n <= 6144 else None)
    if full is not None:
        return full if n <= n_head + n_tail else np.concatenate([full[:n_head], full[n - n_tail:]] if n_tail else [full[:n_head]])
    return np.concatenate([h[:n_head], t[-n_tail:]] if n_tail else [h[:n_head]])

if os.path.exists(CACHE6):
    D6 = pickle.load(open(CACHE6, "rb")); print("loaded cache from Drive")
else:
    local = snapshot_download(repo_id=HF_DATASET_REPO, repo_type="dataset",
                              allow_patterns=["Phase_6_Summ/*", "*/Phase_6_Summ/*"], token=HF_TOKEN)
    def find(name):
        for r, _, fs in os.walk(local):
            if name in fs and "Phase_6" in r: return os.path.join(r, name)
    tr = [r for r in json.load(open(find("train.json"))) if r["summary"].strip()]
    te = json.load(open(find("test.json")))
    def enc(texts, bs=128):
        out = []
        for i in range(0, len(texts), bs):
            out += [e.ids for e in tokenizer.encode_batch([t[:400000] for t in texts[i:i + bs]])]
        return out
    def pack(ids): return dict(n=len(ids), head=np.array(ids[:4096], np.uint16), tail=np.array(ids[-2048:], np.uint16))
    tr_docs, tr_sums = enc([r["document"] for r in tr]), enc([r["summary"] for r in tr])
    te_docs = enc([r["document"] for r in te])
    recs = [dict(id=r["id"], **pack(d), s_len=len(s), summ=np.array(s[:1024], np.uint16)) for r, d, s in zip(tr, tr_docs, tr_sums)]
    perm = np.random.RandomState(0).permutation(len(recs))
    D6 = dict(val=[recs[i] for i in perm[:300]], train=[recs[i] for i in perm[300:]],
              test=[dict(id=r["id"], **pack(d)) for r, d in zip(te, te_docs)])
    pickle.dump(D6, open(CACHE6, "wb")); print("saved cache to Drive")

print("train:", len(D6["train"]), "| val:", len(D6["val"]), "| test:", len(D6["test"]), "| EOS id:", EOS6)
r = D6["train"][0]
print("view check: head-only 1536 ->", len(doc_view(r, 1536)), "| head 1024 + tail 512 ->", len(doc_view(r, 1024, 512)), "| doc length", r["n"])

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

saved cache to Drive
train: 6728 | val: 300 | test: 100 | EOS id: 0
view check: head-only 1536 -> 1536 | head 1024 + tail 512 -> 1536 | doc length 3103


In [ ]:
import math, os, torch, torch.nn as nn, torch.nn.functional as F
KIND = "gqa"          # <-- set per notebook: gqa | strided | mla | encdec | student
assert "RRModel" in globals() and "tokenizer" in globals() and "device" in globals(), "run setup + Phase-3 toolkit first"
P1 = DRIVE_PROJECT_DIR                                    # Phase-1 folder (from the setup cell)
STAGE1_CKPT = "/content/drive/MyDrive/CS781_Capstone_Project_Phase_2/student_kd_stage1/best.pt"

def _load(lm, path):
    ck = torch.load(path, map_location="cpu", weights_only=False)
    sd = ck.get("model", ck.get("model_state_dict", ck)) if isinstance(ck, dict) else ck
    lm.load_state_dict({k.replace("_orig_mod.", "").replace("module.", ""): v for k, v in sd.items()})
    return lm

def build_lm(kind):
    V = tokenizer.get_vocab_size()
    if kind == "gqa":
        return _load(CustomDecoderLM(vocab_size=V, d_model=768, n_layer=10, n_head=12, n_kv_head=4, max_seq_len=2048, tie_weights=True),
                     f"{P1}/gqa_checkpoints/gqa_best.pt"), 768
    if kind == "strided":
        pat = alternating_patterns(10, [(256, 32), (256, 128)])
        return _load(CustomDecoderLM(vocab_size=V, d_model=768, n_layer=10, n_head=12, n_kv_head=4, max_seq_len=2048,
                                     tie_weights=True, layer_patterns=pat), f"{P1}/alt_stride_32_128_checkpoints/best.pt"), 768
    if kind == "mla":
        return _load(CustomDecoderLM(vocab_size=V, d_model=768, n_layer=10, n_head=12, max_seq_len=4096, qk_rope_head_dim=64,
                                     tie_weights=True), f"{P1}/DeepSeek/deepseek_mla_85m_pretrained.pt"), 768
    if kind == "encdec":
        return _load(EncoderDecoderLM(vocab_size=V, d_model=640, n_enc_layer=6, n_dec_layer=6, n_head=10, n_kv_head=5,
                                      max_src_len=1024, max_tgt_len=320, bos_id=tokenizer.token_to_id("<|endoftext|>"),
                                      tie_weights=True), f"{P1}/encdec_6x6_checkpoints/best.pt"), 640
    if kind == "student":
        return _load(CustomDecoderLM(vocab_size=V, d_model=576, n_layer=7, n_head=12, n_kv_head=4, max_seq_len=2048,
                                     tie_weights=True), STAGE1_CKPT), 576
    raise ValueError(kind)

class Backbone(RRModel):
    """Phase-1 LM + a features() method. Task heads are attached by each phase (model.head = ...)."""
    def __init__(self, lm, d, kind):
        super().__init__(lm, d, 2, 0.1); del self.head
        self.kind, self.d = kind, d
    def features(self, ids, lens):                       # ids (chunks, T) right-padded, lens (chunks,) -> (chunks, 2d)
        if self.kind == "encdec":                        # encoder takes no mask: one chunk at a time
            out = []
            for j in range(ids.shape[0]):
                n = int(lens[j]); H = self.lm.encode(ids[j:j + 1, :n])[0].float()
                out.append(torch.cat([H[n - 1], H.mean(0)], -1))
            return torch.stack(out)
        H = self.hidden(ids); B, T, _ = H.shape
        mask = (torch.arange(T, device=H.device)[None] < lens[:, None]).unsqueeze(-1)
        return torch.cat([H[torch.arange(B, device=H.device), lens - 1].float(), (H.float() * mask).sum(1) / lens[:, None]], -1)

def make_backbone(kind=None):
    lm, d = build_lm(kind or KIND); return Backbone(lm, d, kind or KIND).to(device)

# ---------- adapters (own implementations) ----------
class LoRALinear(nn.Module):
    def __init__(self, base, r, alpha):
        super().__init__(); self.base, self.r, self.scale = base, r, alpha / r
        self.A = nn.Parameter(torch.empty(r, base.in_features)); nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.B = nn.Parameter(torch.zeros(base.out_features, r))
    def forward(self, x): return self.base(x) + self.scale * F.linear(F.linear(x, self.A), self.B)

class LoRMAPlusLinear(nn.Module):
    """W = ((alpha/r) B A + I) W0 on the output side. Identity at init, mergeable."""
    def __init__(self, base, r, alpha):
        super().__init__(); self.base, self.r, self.scale = base, r, alpha / r
        self.A = nn.Parameter(torch.empty(r, base.out_features)); nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.B = nn.Parameter(torch.zeros(base.out_features, r))
    def forward(self, x):
        y = self.base(x); return y + self.scale * F.linear(F.linear(y, self.A), self.B)

TARGETS = {k: ("q_proj", "k_proj", "v_proj", "out_proj") for k in ["gqa", "strided", "encdec", "student"]}
TARGETS["mla"] = ("q_up_proj", "kv_down_proj", "kv_up_proj", "out_proj")
def layers_of(bb): return bb.lm.enc_layers if bb.kind == "encdec" else bb.lm.layers

def inject(bb, method, r, alpha):
    """Freeze the Phase-1 LM and add adapters. Heads attached outside bb.lm stay trainable."""
    for p in bb.lm.parameters(): p.requires_grad_(False)
    cls = {"lora": LoRALinear, "lorma": LoRMAPlusLinear}[method]
    for layer in layers_of(bb):
        for nm in TARGETS[bb.kind]:
            base = getattr(layer.attn, nm); setattr(layer.attn, nm, cls(base, r, alpha).to(base.weight.device))
    return bb

def unfreeze_top(bb, k):
    """Partial (layer-wise) fine-tuning: only the top k layers (+ final norm for decoders) train."""
    for p in bb.lm.parameters(): p.requires_grad_(False)
    for layer in list(layers_of(bb))[-k:]:
        for p in layer.parameters(): p.requires_grad_(True)
    if bb.kind != "encdec":
        for p in bb.lm.norm.parameters(): p.requires_grad_(True)
    return bb

def n_params(m): return sum(p.numel() for p in m.parameters() if p.requires_grad), sum(p.numel() for p in m.parameters())

# ---------- distillation losses ----------
def kd_multiclass(s, t, T=2.0): return F.kl_div(F.log_softmax(s / T, -1), F.softmax(t / T, -1), reduction="batchmean") * T * T
def kd_multilabel(s, t, T=2.0): return F.binary_cross_entropy_with_logits(s / T, torch.sigmoid(t / T)) * T * T

# ---------- self-test: shapes + adapters are the identity at init ----------
CORE_SELFTEST = True
if CORE_SELFTEST:
    ids = torch.randint(0, tokenizer.get_vocab_size(), (2, 64), device=device); lens = torch.tensor([50, 64], device=device)
    for method in ["lorma", "lora"]:
        bb = make_backbone().eval()
        with torch.no_grad(): f0 = bb.features(ids, lens)
        inject(bb, method, 8, 16)
        with torch.no_grad(): f1 = bb.features(ids, lens)
        assert f0.shape == (2, 2 * bb.d) and (f0 - f1).abs().max().item() < 1e-3, f"{method}: not identity at init"
        tr, tot = n_params(bb); print(f"[{KIND}] {method}: identity OK | trainable {tr/1e6:.2f}M of {tot/1e6:.1f}M (adapters only, no head)")
        del bb; torch.cuda.empty_cache()
    bb = unfreeze_top(make_backbone(), 4); tr, tot = n_params(bb); print(f"[{KIND}] partial top-4: trainable {tr/1e6:.2f}M of {tot/1e6:.1f}M"); del bb
    print(f"✅ core ready for KIND={KIND}")

[gqa] lorma: identity OK | trainable 0.33M of 87.8M (adapters only, no head)
[gqa] lora: identity OK | trainable 0.41M of 87.9M (adapters only, no head)
[gqa] partial top-4: trainable 25.17M of 87.5M
✅ core ready for KIND=gqa


In [ ]:
for n in ["tokenizer", "device", "HF_DATASET_REPO", "HF_TOKEN", "RRModel", "build_lm", "make_backbone", "Backbone", "inject", "KIND"]:
    print("✅" if n in globals() else "❌ MISSING", n)

✅ tokenizer
✅ device
✅ HF_DATASET_REPO
✅ HF_TOKEN
✅ RRModel
✅ build_lm
✅ make_backbone
✅ Backbone
✅ inject
✅ KIND


In [ ]:
import inspect, torch
bb = make_backbone(); lm = bb.lm; cls = type(lm)
print("class:", cls.__name__, "| KIND:", KIND)
print("public methods:", [m for m in dir(cls) if not m.startswith("_") and callable(getattr(cls, m))])
for nm in ["forward", "generate", "encode", "decode"]:
    f = getattr(cls, nm, None)
    if f is None: continue
    print(f"\n{nm}{inspect.signature(f)}\n  names used: {f.__code__.co_names}")
ids = torch.randint(0, 100, (1, 16), device=device)
with torch.no_grad():
    try:
        o = lm(ids); parts = o if isinstance(o, tuple) else (o,)
        print("\nlm(ids) ->", type(o).__name__, [getattr(t, "shape", type(t).__name__) for t in parts])
        if isinstance(o, tuple) and len(o) > 1:
            print("2nd output:", type(o[1]).__name__, "len", len(o[1]) if hasattr(o[1], "__len__") else "-")
            o2 = lm(ids[:, -1:], past_key_values=o[1])
            print("cached step ->", [getattr(t, "shape", type(t).__name__) for t in (o2 if isinstance(o2, tuple) else (o2,))])
    except Exception as e:
        print("forward probe failed:", repr(e)[:400])

class: CustomDecoderLM | KIND: gqa
public methods: ['add_module', 'apply', 'bfloat16', 'buffers', 'children', 'compile', 'cpu', 'cuda', 'double', 'eval', 'extra_repr', 'float', 'forward', 'get_buffer', 'get_extra_state', 'get_parameter', 'get_submodule', 'half', 'ipu', 'load_state_dict', 'modules', 'mtia', 'named_buffers', 'named_children', 'named_modules', 'named_parameters', 'parameters', 'register_backward_hook', 'register_buffer', 'register_forward_hook', 'register_forward_pre_hook', 'register_full_backward_hook', 'register_full_backward_pre_hook', 'register_load_state_dict_post_hook', 'register_load_state_dict_pre_hook', 'register_module', 'register_parameter', 'register_state_dict_post_hook', 'register_state_dict_pre_hook', 'requires_grad_', 'set_extra_state', 'set_submodule', 'share_memory', 'state_dict', 'to', 'to_empty', 'train', 'type', 'xpu', 'zero_grad']

forward(self, input_ids, labels=None, past_key_values=None)
  names used: ('shape', 'token_embedding', 'enumerate', 'lay

In [ ]:
import torch, torch.nn.functional as F
bb = make_backbone().eval(); lm = bb.lm
V = tokenizer.get_vocab_size(); ids = torch.randint(0, V, (1, 40), device=device)
pick_lg = lambda o: next(v for v in o.values() if torch.is_tensor(v) and v.dim() == 3)
with torch.no_grad():
    o = lm(ids); print("keys:", {k: (tuple(v.shape) if torch.is_tensor(v) else type(v).__name__) for k, v in o.items()})
    full = pick_lg(o)
    pk_key = next((k for k, v in o.items() if "past" in k or "cache" in k or isinstance(v, (tuple, list))), None)
    print("cache key:", pk_key)
    o1 = lm(ids[:, :39]); o2 = lm(ids[:, 39:40], past_key_values=o1[pk_key])
    print("cached step vs full forward, max logit diff: %.2e" % (pick_lg(o2)[:, -1] - full[:, 39]).abs().max().item())
    lab = ids.clone(); lab[:, :20] = -100
    L = lm(ids, labels=lab); loss = next(v for k, v in L.items() if "loss" in k)
    lg = pick_lg(L)
    man = F.cross_entropy(lg[:, :-1].reshape(-1, V).float(), lab[:, 1:].reshape(-1), ignore_index=-100)
    print("model loss %.4f | manual shifted CE %.4f" % (loss.item(), man.item()))

keys: {'loss': 'NoneType', 'logits': (1, 40, 32000), 'past_key_values': 'tuple'}
cache key: past_key_values
cached step vs full forward, max logit diff: 1.72e-05
model loss 12.9054 | manual shifted CE 12.9054


In [ ]:
import os, time, json, numpy as np, torch
try: from rouge_score import rouge_scorer
except ImportError:
    import subprocess, sys; subprocess.run([sys.executable, "-m", "pip", "install", "-q", "rouge_score"]); from rouge_score import rouge_scorer
assert KIND != "encdec", "decoder cell: enc-dec gets its own"
MAXLEN = 4096 if KIND == "mla" else 2048
CFG = dict(doc_head=2048, doc_tail=1024, sum_max=960) if KIND == "mla" else dict(doc_head=1024, doc_tail=480, sum_max=512)
BS, ACC = (4, 2) if KIND == "mla" else (8, 1)
SEP = tokenizer.encode("\n\nSUMMARY:\n").ids
assert len(SEP) + CFG["doc_head"] + CFG["doc_tail"] + CFG["sum_max"] + 1 <= MAXLEN
scorer = rouge_scorer.RougeScorer(["rouge1", "rougeL"], use_stemmer=True)
R6 = globals().get("R6", {})

def make_prompt(r): return doc_view(r, CFG["doc_head"], CFG["doc_tail"]).astype(np.int64).tolist() + SEP
def make_ex(r):
    s = r["summ"][:CFG["sum_max"]].astype(np.int64).tolist()
    if r["s_len"] <= CFG["sum_max"]: s.append(EOS6)          # teach stopping only when the whole summary fit
    return make_prompt(r), s
def batchify(exs):
    T = max(len(p) + len(s) for p, s in exs)
    ids = torch.zeros(len(exs), T, dtype=torch.long); lab = torch.full((len(exs), T), -100, dtype=torch.long)
    for i, (p, s) in enumerate(exs):
        x = p + s; ids[i, :len(x)] = torch.tensor(x); lab[i, len(p):len(x)] = torch.tensor(s)
    return ids.to(device), lab.to(device)

@torch.no_grad()
def val_loss(bb, n=None):
    bb.eval(); vs = D6["val"][:n] if n else D6["val"]; out = []
    for i in range(0, len(vs), BS):
        ids, lab = batchify([make_ex(r) for r in vs[i:i + BS]])
        with torch.autocast("cuda", dtype=torch.bfloat16): out.append(bb.lm(ids, labels=lab)["loss"].item())
    bb.train(); return float(np.mean(out))

@torch.no_grad()
def generate(bb, prompt, max_new, nrep=4):
    lm = bb.lm.eval(); out, seen = [], {}
    with torch.autocast("cuda", dtype=torch.bfloat16):
        o = lm(torch.tensor([prompt], device=device)); pk = o["past_key_values"]; lg = o["logits"][0, -1].float()
        for _ in range(max_new):
            key = tuple(out[-(nrep - 1):]) if nrep and len(out) >= nrep - 1 else None
            if key is not None and key in seen: lg[list(seen[key])] = -float("inf")
            t = int(lg.argmax())
            if t == EOS6: break
            if key is not None: seen.setdefault(key, set()).add(t)
            out.append(t)
            o = lm(torch.tensor([[t]], device=device), past_key_values=pk); pk = o["past_key_values"]; lg = o["logits"][0, -1].float()
    return out

def rouge_eval(bb, n=40, max_new=None, show=0):
    r1, rl, wl = [], [], []
    for j, r in enumerate(D6["val"][:n]):
        hyp = tokenizer.decode(generate(bb, make_prompt(r), max_new or CFG["sum_max"]))
        ref = tokenizer.decode(r["summ"].astype(np.int64).tolist())          # reference truncated at 1024 tokens (proxy)
        sc = scorer.score(ref, hyp); r1.append(sc["rouge1"].fmeasure); rl.append(sc["rougeL"].fmeasure); wl.append(len(hyp.split()))
        if j < show: print(f"\n--- val doc {j} ---\nHYP: {hyp[:600]}\nREF: {ref[:600]}")
    return dict(rouge1=float(np.mean(r1)), rougeL=float(np.mean(rl)), words=float(np.mean(wl)))

def train_sum(bb, name, lr, epochs=3, wd=0.01, patience=3, debug=False):
    params = [p for p in bb.parameters() if p.requires_grad]; assert params, "nothing trainable"
    tr = D6["train"][:64] if debug else D6["train"]; epochs = 1 if debug else epochs
    steps = epochs * (len(tr) // (BS * ACC)); warm = max(1, int(0.06 * steps)); ev = max(1, steps // (2 * epochs))
    opt = torch.optim.AdamW(params, lr=lr, weight_decay=wd)
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: min((s + 1) / warm, max(0.0, (steps - s) / max(1, steps - warm))))
    path = f"{P6_DIR}/sm_{name}_best.pt"; best, bad, step, run, t0, stop = 1e9, 0, 0, [], time.time(), False
    ntr, ntot = n_params(bb); print(f"[{name}] trainable {ntr/1e6:.2f}M / {ntot/1e6:.1f}M | steps {steps} | eval every {ev}")
    torch.cuda.reset_peak_memory_stats(); bb.train()
    for ep in range(epochs):
        order = np.random.RandomState(ep).permutation(len(tr)); k = 0
        while k + BS * ACC <= len(tr) and step < steps and not stop:
            for _ in range(ACC):
                ids, lab = batchify([make_ex(tr[i]) for i in order[k:k + BS]]); k += BS
                with torch.autocast("cuda", dtype=torch.bfloat16): loss = bb.lm(ids, labels=lab)["loss"] / ACC
                loss.backward(); run.append(loss.item() * ACC)
            torch.nn.utils.clip_grad_norm_(params, 1.0); opt.step(); sched.step(); opt.zero_grad(set_to_none=True); step += 1
            if step % ev == 0 or step == steps:
                vl = val_loss(bb, 16 if debug else None)
                print(f"step {step}/{steps} | train {np.mean(run):.3f} | val {vl:.3f} | {(time.time()-t0)/60:.1f} min"); run = []
                if vl < best:
                    best, bad = vl, 0; torch.save({n: p.detach().cpu() for n, p in bb.named_parameters() if p.requires_grad}, path)
                else:
                    bad += 1; stop = bad >= patience
    bb.load_state_dict(torch.load(path), strict=False)
    m = rouge_eval(bb, n=6 if debug else 40, show=1 if debug else 0)
    R6[name] = dict(val_loss=best, **m, trainable_M=ntr / 1e6, peak_gb=torch.cuda.max_memory_allocated() / 1e9, minutes=(time.time() - t0) / 60)
    print(f"\n✅ [{name}] best val loss {best:.3f} | ROUGE-1 {m['rouge1']:.4f} | ROUGE-L {m['rougeL']:.4f} | avg words {m['words']:.0f} | peak {R6[name]['peak_gb']:.1f} GB")
    return bb

def write_pred_sum(bb, name, max_new=None):
    preds = []; t0 = time.time()
    for i, r in enumerate(D6["test"]):
        h = tokenizer.decode(generate(bb, make_prompt(r), max_new or CFG["sum_max"])).strip()
        preds.append(dict(id=r["id"], summary=h or tokenizer.decode(doc_view(r, 150).astype(np.int64).tolist())))
        if (i + 1) % 25 == 0: print(f"{i+1}/100 | {(time.time()-t0)/60:.1f} min")
    assert len(preds) == 100 and len({p["id"] for p in preds}) == 100
    p = f"{P6_DIR}/prediction_{name}.json"; json.dump(preds, open(p, "w")); print("✅ wrote", p, "-> rename to prediction.json, zip, upload")
print("✅ summarisation toolkit ready | KIND =", KIND, "| context", MAXLEN, "|", CFG, "| BS", BS, "x ACC", ACC)

✅ summarisation toolkit ready | KIND = gqa | context 2048 | {'doc_head': 1024, 'doc_tail': 480, 'sum_max': 512} | BS 8 x ACC 1


In [ ]:
bb = make_backbone(); inject(bb, "lorma", 64, 128); bb = train_sum(bb, "gqa_lorma", lr=5e-4)
write_pred_sum(bb, "gqa_lorma")

[gqa_lorma] trainable 2.62M / 90.1M | steps 2523 | eval every 420
step 420/2523 | train 1.912 | val 1.847 | 2.6 min
step 840/2523 | train 1.852 | val 1.826 | 5.3 min
step 1260/2523 | train 1.789 | val 1.813 | 7.9 min
step 1680/2523 | train 1.786 | val 1.803 | 10.6 min
step 2100/2523 | train 1.725 | val 1.802 | 13.2 min
step 2520/2523 | train 1.733 | val 1.797 | 15.9 min
step 2523/2523 | train 1.636 | val 1.797 | 16.0 min

✅ [gqa_lorma] best val loss 1.797 | ROUGE-1 0.4914 | ROUGE-L 0.2768 | avg words 383 | peak 36.0 GB
25/100 | 3.2 min
50/100 | 6.4 min
75/100 | 9.8 min
100/100 | 13.3 min
✅ wrote /content/drive/MyDrive/CS781_Capstone_Project_Phase_6/prediction_gqa_lorma.json -> rename to prediction.json, zip, upload


In [ ]:
for L in (128, 256, 512): print(L, rouge_eval(bb, n=40, max_new=L))
del bb; torch.cuda.empty_cache()

128 {'rouge1': 0.2606377245474073, 'rougeL': 0.17509594870287432, 'words': 113.25}
256 {'rouge1': 0.3949517891030153, 'rougeL': 0.2401593941372842, 'words': 223.1}
512 {'rouge1': 0.4913611402824115, 'rougeL': 0.27683663003628356, 'words': 383.425}


In [ ]:
bb = make_backbone(); bb = train_sum(bb, "gqa_full", lr=5e-5, epochs=2)
write_pred_sum(bb, "gqa_full"); del bb; torch.cuda.empty_cache()

[gqa_full] trainable 87.51M / 87.5M | steps 1682 | eval every 420
step 420/1682 | train 1.879 | val 1.808 | 2.7 min
step 840/1682 | train 1.803 | val 1.768 | 5.4 min
step 1260/1682 | train 1.581 | val 1.765 | 8.1 min
step 1680/1682 | train 1.577 | val 1.760 | 10.8 min
step 1682/1682 | train 1.673 | val 1.760 | 11.0 min

✅ [gqa_full] best val loss 1.760 | ROUGE-1 0.5006 | ROUGE-L 0.2784 | avg words 377 | peak 38.8 GB
25/100 | 2.6 min
50/100 | 5.3 min
75/100 | 7.9 min
100/100 | 10.6 min
✅ wrote /content/drive/MyDrive/CS781_Capstone_Project_Phase_6/prediction_gqa_full.json -> rename to prediction.json, zip, upload


In [ ]:
bb = make_backbone(); inject(bb, "lora", 64, 128); bb = train_sum(bb, "gqa_lora", lr=5e-4, epochs=2)
write_pred_sum(bb, "gqa_lora"); del bb; torch.cuda.empty_cache()

[gqa_lora] trainable 3.28M / 90.8M | steps 1682 | eval every 420
step 420/1682 | train 1.901 | val 1.835 | 2.7 min
step 840/1682 | train 1.838 | val 1.808 | 5.4 min
step 1260/1682 | train 1.758 | val 1.798 | 8.1 min
step 1680/1682 | train 1.752 | val 1.792 | 10.8 min
step 1682/1682 | train 1.884 | val 1.792 | 10.9 min

✅ [gqa_lora] best val loss 1.792 | ROUGE-1 0.4950 | ROUGE-L 0.2674 | avg words 378 | peak 36.0 GB
25/100 | 3.6 min
50/100 | 7.3 min
75/100 | 10.9 min
100/100 | 14.6 min
✅ wrote /content/drive/MyDrive/CS781_Capstone_Project_Phase_6/prediction_gqa_lora.json -> rename to prediction.json, zip, upload


In [ ]:
import torch.nn as nn, torch.nn.functional as F
class AdapterLinear(nn.Module):
    def __init__(self, base, m):
        super().__init__(); self.base = base
        self.down = nn.Linear(base.out_features, m); self.up = nn.Linear(m, base.out_features)
        nn.init.zeros_(self.up.weight); nn.init.zeros_(self.up.bias)
    def forward(self, x): y = self.base(x); return y + self.up(F.gelu(self.down(y)))
def inject_houlsby(bb, m=128):
    for p in bb.lm.parameters(): p.requires_grad_(False)
    for layer in layers_of(bb):
        for owner, nm in ((layer.attn, "out_proj"), (layer.ffn, "w2")):
            base = getattr(owner, nm); setattr(owner, nm, AdapterLinear(base, m).to(base.weight.device))
    return bb

bb = make_backbone(); inject_houlsby(bb); bb = train_sum(bb, "gqa_houlsby", lr=5e-4, epochs=2)
write_pred_sum(bb, "gqa_houlsby"); del bb; torch.cuda.empty_cache()

[gqa_houlsby] trainable 3.95M / 91.5M | steps 1682 | eval every 420
step 420/1682 | train 1.900 | val 1.837 | 2.5 min
step 840/1682 | train 1.839 | val 1.808 | 5.0 min
step 1260/1682 | train 1.767 | val 1.798 | 7.6 min
step 1680/1682 | train 1.760 | val 1.791 | 10.1 min
step 1682/1682 | train 1.892 | val 1.791 | 10.2 min

✅ [gqa_houlsby] best val loss 1.791 | ROUGE-1 0.4791 | ROUGE-L 0.2577 | avg words 395 | peak 33.1 GB
25/100 | 3.1 min
50/100 | 6.3 min
75/100 | 9.5 min
100/100 | 12.9 min
✅ wrote /content/drive/MyDrive/CS781_Capstone_Project_Phase_6/prediction_gqa_houlsby.json -> rename to prediction.json, zip, upload


In [ ]:
def load_teacher(name, method=None):
    t = make_backbone("gqa")
    if method: inject(t, method, 64, 128)
    t.load_state_dict(torch.load(f"{P6_DIR}/sm_{name}_best.pt"), strict=False)
    t.eval()
    for p in t.parameters(): p.requires_grad_(False)
    return t

def train_kd(student, teacher, name, lr=1e-4, alpha=0.5, T=2.0, epochs=2, wd=0.01, patience=3, debug=False):
    params = [p for p in student.parameters() if p.requires_grad]
    tr = D6["train"][:64] if debug else D6["train"]; epochs = 1 if debug else epochs
    steps = epochs * (len(tr) // (BS * ACC)); warm = max(1, int(0.06 * steps)); ev = max(1, steps // (2 * epochs))
    opt = torch.optim.AdamW(params, lr=lr, weight_decay=wd)
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: min((s + 1) / warm, max(0.0, (steps - s) / max(1, steps - warm))))
    path = f"{P6_DIR}/sm_{name}_best.pt"; best, bad, step, run, t0, stop = 1e9, 0, 0, [], time.time(), False
    ntr, ntot = n_params(student); print(f"[{name}] student trainable {ntr/1e6:.1f}M | steps {steps} | eval every {ev}")
    torch.cuda.reset_peak_memory_stats(); student.train()
    for ep in range(epochs):
        order = np.random.RandomState(ep).permutation(len(tr)); k = 0
        while k + BS * ACC <= len(tr) and step < steps and not stop:
            for _ in range(ACC):
                ids, lab = batchify([make_ex(tr[i]) for i in order[k:k + BS]]); k += BS
                with torch.autocast("cuda", dtype=torch.bfloat16):
                    with torch.no_grad(): tl = teacher.lm(ids)["logits"][:, :-1]
                    sl = student.lm(ids)["logits"][:, :-1]
                m = lab[:, 1:] != -100
                s, t = sl[m].float(), tl[m].float()
                ce = F.cross_entropy(s, lab[:, 1:][m])
                kd = F.kl_div(F.log_softmax(s / T, -1), F.softmax(t / T, -1), reduction="batchmean") * T * T
                ((1 - alpha) * ce + alpha * kd).div(ACC).backward(); run.append(ce.item())
            torch.nn.utils.clip_grad_norm_(params, 1.0); opt.step(); sched.step(); opt.zero_grad(set_to_none=True); step += 1
            if step % ev == 0 or step == steps:
                vl = val_loss(student, 16 if debug else None)
                print(f"step {step}/{steps} | train CE {np.mean(run):.3f} | val CE {vl:.3f} | {(time.time()-t0)/60:.1f} min"); run = []
                if vl < best:
                    best, bad = vl, 0; torch.save({n: p.detach().cpu() for n, p in student.named_parameters() if p.requires_grad}, path)
                else:
                    bad += 1; stop = bad >= patience
    student.load_state_dict(torch.load(path), strict=False)
    mt = rouge_eval(student, n=6 if debug else 40)
    R6[name] = dict(val_loss=best, **mt, trainable_M=ntr / 1e6, peak_gb=torch.cuda.max_memory_allocated() / 1e9, minutes=(time.time() - t0) / 60)
    print(f"\n✅ [{name}] best val CE {best:.3f} | ROUGE-1 {mt['rouge1']:.4f} | ROUGE-L {mt['rougeL']:.4f} | avg words {mt['words']:.0f}")
    return student

In [ ]:
teacher = load_teacher("gqa_full")                     # or: load_teacher("gqa_lorma", "lorma")
st = make_backbone("student"); train_kd(st, teacher, "dbg_kd", debug=False)     # smoke test first

[dbg_kd] student trainable 43.2M | steps 1682 | eval every 420
step 420/1682 | train CE 2.245 | val CE 2.119 | 2.8 min
step 840/1682 | train CE 2.091 | val CE 2.054 | 5.7 min
step 1260/1682 | train CE 1.882 | val CE 2.028 | 8.5 min
step 1680/1682 | train CE 1.880 | val CE 2.015 | 11.3 min
step 1682/1682 | train CE 2.004 | val CE 2.015 | 11.4 min

✅ [dbg_kd] best val CE 2.015 | ROUGE-1 0.4695 | ROUGE-L 0.2537 | avg words 386


Backbone(
  (lm): CustomDecoderLM(
    (token_embedding): Embedding(32000, 576)
    (layers): ModuleList(
      (0-6): 7 x TransformerBlock(
        (attention_norm): RMSNorm()
        (attn): GroupedQueryAttention(
          (q_proj): Linear(in_features=576, out_features=576, bias=False)
          (k_proj): Linear(in_features=576, out_features=192, bias=False)
          (v_proj): Linear(in_features=576, out_features=192, bias=False)
          (out_proj): Linear(in_features=576, out_features=576, bias=False)
        )
        (ffn_norm): RMSNorm()
        (ffn): SwiGLU(
          (w1): Linear(in_features=576, out_features=1536, bias=False)
          (w2): Linear(in_features=1536, out_features=576, bias=False)
          (w3): Linear(in_features=576, out_features=1536, bias=False)
        )
      )
    )
    (norm): RMSNorm()
    (lm_head): Linear(in_features=576, out_features=32000, bias=False)
  )
  (drop): Dropout(p=0.1, inplace=False)
)

In [ ]:
st = make_backbone("student"); st = train_kd(st, teacher, "gqa_student_kd")
write_pred_sum(st, "gqa_student_kd"); del st; torch.cuda.empty_cache()

[gqa_student_kd] student trainable 43.2M | steps 1682 | eval every 420
step 420/1682 | train CE 2.245 | val CE 2.119 | 2.8 min
step 840/1682 | train CE 2.091 | val CE 2.054 | 5.7 min
step 1260/1682 | train CE 1.882 | val CE 2.028 | 8.5 min
step 1680/1682 | train CE 1.880 | val CE 2.015 | 11.3 min
step 1682/1682 | train CE 2.004 | val CE 2.015 | 11.4 min

✅ [gqa_student_kd] best val CE 2.015 | ROUGE-1 0.4695 | ROUGE-L 0.2537 | avg words 386
25/100 | 1.8 min
50/100 | 3.6 min
75/100 | 5.3 min
100/100 | 7.2 min
✅ wrote /content/drive/MyDrive/CS781_Capstone_Project_Phase_6/prediction_gqa_student_kd.json -> rename to prediction.json, zip, upload


In [ ]:
st = make_backbone("student"); st = train_sum(st, "gqa_student_ce", lr=1e-4, epochs=2)
del st; torch.cuda.empty_cache()

[gqa_student_ce] trainable 43.21M / 43.2M | steps 1682 | eval every 420
step 420/1682 | train 2.275 | val 2.146 | 1.8 min
step 840/1682 | train 2.130 | val 2.072 | 3.6 min
step 1260/1682 | train 1.887 | val 2.053 | 5.5 min
step 1680/1682 | train 1.876 | val 2.037 | 7.3 min
step 1682/1682 | train 1.991 | val 2.037 | 7.4 min

✅ [gqa_student_ce] best val loss 2.037 | ROUGE-1 0.4670 | ROUGE-L 0.2557 | avg words 379 | peak 28.2 GB


In [ ]:
import pandas as pd; print(pd.DataFrame(R6).T.round(4))

                val_loss  rouge1  rougeL     words  trainable_M  peak_gb  \
dbg               2.1043  0.3441  0.1821  382.6667       2.6214  34.8546   
gqa_lorma         1.7972  0.4914  0.2768  383.4250       2.6214  36.0286   
gqa_full          1.7598  0.5006  0.2784  376.9750      87.5067  38.8286   
gqa_lora          1.7922  0.4950  0.2674  378.5000       3.2768  35.9728   
gqa_houlsby       1.7909  0.4791  0.2577  395.1250       3.9501  33.1306   
dbg_kd            2.5305  0.4357  0.2238  437.5000      43.2132  29.4048   
gqa_student_kd    2.0147  0.4695  0.2537  385.5000      43.2132  29.6433   
gqa_student_ce    2.0366  0.4670  0.2557  378.5750      43.2132  28.2494   

                minutes  
dbg              1.0646  
gqa_lorma       21.6329  
gqa_full        15.0769  
gqa_lora        16.6248  
gqa_houlsby     15.5720  
dbg_kd           0.5968  
gqa_student_kd  14.4899  
gqa_student_ce  10.3655  
